# MadGraph NWA vs Full Approximation Comparison

## Overview
This notebook compares **Narrow Width Approximation (NWA)** and **full** calculations in MadGraph **v3.7.0**.

## Scope
- **Processes**: WZjj_EW, WZjj_QCD
- **Version**: MadGraph v3.7.0 only
- **Analysis**: Expected event yields in SR, Low-Mjj CR, and WZ CR using `compute_expected_events_mg_by_region`

## Data Status
- ✓ **v3.7.0 NWA**: Included
- ✓ **v3.7.0 Full**: Included
- Note: Missing samples will display as N/A in output tables

## Workflow
1. Import and configure analysis environment
2. Define sample paths and check file availability
3. Compute expected events for each configuration
4. Generate comparison tables showing yield differences
5. Summarize data completion status

## Environment Setup & Imports

In [1]:
import os
import numpy as np
from pathlib import Path
from tqdm import tqdm

# Import selection yields analysis tools
# ROOT initialization is handled automatically on first use
from selection.yields import compute_expected_events_mg_by_region
from selection.yields import print_expected_event_counts_table

## Analysis Parameters & Configuration

In [2]:
# === Global Configuration ===
luminosity_fb_inv = 139.0
num_workers = None  # Auto-detect CPU count
chunks_per_file = 16

# === Directory structure ===
base_sample_dir = "/home/r10222035/longitudinal_W/Sample"
mg_base_dir = f"{base_sample_dir}/MG5"

# === Processes to analyze ===
# For first version, comparing WZjj_EW and WZjj_QCD only
processes = ["WZjj_EW", "WZjj_QCD"]

print(f"Luminosity: {luminosity_fb_inv} fb⁻¹")
print(f"Workers: {num_workers} (auto-detect)")
print(f"Processes to analyze: {processes}")
print(f"MadGraph base directory: {mg_base_dir}")

Luminosity: 139.0 fb⁻¹
Workers: None (auto-detect)
Processes to analyze: ['WZjj_EW', 'WZjj_QCD']
MadGraph base directory: /home/r10222035/longitudinal_W/Sample/MG5


## Sample Configuration & Availability Check

In [ ]:
# === Define sample configurations ===
# Structure: {sample_name: {'root': path, 'banner': path}}

def build_sample_inputs(processes_list, approximation):
    """Build sample_inputs dict for v3.7.0 samples with fixed tag_1 files."""
    inputs = {}
    for proc in processes_list:
        if approximation == "NWA":
            suffix = "-NWA-v370"
        elif approximation == "full":
            suffix = "-full-v370"
        else:
            continue

        sample_dir = f"{mg_base_dir}/{proc}{suffix}/Events/run_01"
        root_file = f"{sample_dir}/tag_1_delphes_events.root"
        banner_file = f"{sample_dir}/run_01_tag_1_banner.txt"

        label = f"{proc}_v3.7.0_{approximation}"
        inputs[label] = {
            'root': root_file,
            'banner': banner_file,
            'sample_name': proc,
            'version': "v3.7.0",
            'approximation': approximation,
        }

    return inputs

# Build configurations (v3.7.0 only)
nwa_v370 = build_sample_inputs(processes, "NWA")
full_v370 = build_sample_inputs(processes, "full")

print("Sample configurations defined:")
print(f"  v3.7.0 NWA: {len(nwa_v370)} items")
print(f"  v3.7.0 Full: {len(full_v370)} items")

Sample configurations defined:
  v3.7.0 NWA: 2 items
  v3.7.0 Full: 2 items
  Event files: fixed to tag_1


In [4]:
# === Check file availability ===
def check_sample_availability(sample_dict):
    """
    Check if all required files exist for sample dict.
    Returns tuple: (available_dict, diagnostics_dict)
    """
    available = {}
    diagnostics = {}

    for label, config in sample_dict.items():
        root_exists = os.path.isfile(config['root'])
        banner_exists = os.path.isfile(config['banner'])

        status = "✓ OK"
        missing = []
        if not root_exists:
            missing.append("root")
        if not banner_exists:
            missing.append("banner")
        if missing:
            status = f"⚠ Missing: {', '.join(missing)}"

        diagnostics[label] = {
            'version': config['version'],
            'approximation': config['approximation'],
            'sample_name': config['sample_name'],
            'root_exists': root_exists,
            'banner_exists': banner_exists,
            'status': status,
        }

        if root_exists and banner_exists:
            available[label] = config

    return available, diagnostics

# Run availability checks (v3.7.0 only)
nwa_v370_avail, diag_v370_nwa = check_sample_availability(nwa_v370)
full_v370_avail, diag_v370_full = check_sample_availability(full_v370)

# Print diagnostics
all_diag = {**diag_v370_nwa, **diag_v370_full}
print("\n" + "="*90)
print("SAMPLE AVAILABILITY CHECK")
print("="*90)
print(f"{'Label':<28} {'Version':<12} {'Approx':<8} {'Status':<40}")
print("-"*90)
for label in sorted(all_diag.keys()):
    diag = all_diag[label]
    print(f"{label:<28} {diag['version']:<12} {diag['approximation']:<8} {diag['status']:<40}")

print("="*90)
print(f"\nSummary:")
print(f"  v3.7.0 NWA: {len(nwa_v370_avail)}/{len(nwa_v370)} available")
print(f"  v3.7.0 Full: {len(full_v370_avail)}/{len(full_v370)} available")


SAMPLE AVAILABILITY CHECK
Label                        Version      Approx   Status                                  
------------------------------------------------------------------------------------------
WZjj_EW_v3.7.0_NWA           v3.7.0       NWA      ✓ OK                                    
WZjj_EW_v3.7.0_full          v3.7.0       full     ✓ OK                                    
WZjj_QCD_v3.7.0_NWA          v3.7.0       NWA      ✓ OK                                    
WZjj_QCD_v3.7.0_full         v3.7.0       full     ✓ OK                                    

Summary:
  v3.7.0 NWA: 2/2 available
  v3.7.0 Full: 2/2 available


## Yield Calculations

Computing expected events for all available samples using `compute_expected_events_mg_by_region`.

In [5]:
# === MadGraph 3.7.0 NWA ===
print("\n" + "="*90)
print("COMPUTING: MadGraph v3.7.0 NWA")
print("="*90)

if len(nwa_v370_avail) > 0:
    results_nwa_v370 = compute_expected_events_mg_by_region(
        sample_inputs=nwa_v370_avail,
        luminosity_fb_inv=luminosity_fb_inv,
        num_workers=num_workers,
        chunks_per_file=chunks_per_file,
    )
    print_expected_event_counts_table(
        results_nwa_v370,
        luminosity_fb_inv,
        title_prefix="MadGraph v3.7.0 NWA Expected Event Counts",
        extended_mode=True,
    )
else:
    results_nwa_v370 = {}
    print("⚠ No available samples for v3.7.0 NWA")


COMPUTING: MadGraph v3.7.0 NWA
[Yields] Processing all samples with file + chunk parallelism...


[1/2] WZjj_EW_v3.7.0_NWA: Processing 1 file(s) with 48 worker(s), 16 chunk(s) per file...


    WZjj_EW_v3.7.0_NWA: 100%|██████████| 16/16 [00:01<00:00, 12.87it/s]


  ✓ [WZjj_EW_v3.7.0_NWA] done -> Total=10000 events, passed=573

[2/2] WZjj_QCD_v3.7.0_NWA: Processing 1 file(s) with 48 worker(s), 16 chunk(s) per file...


    WZjj_QCD_v3.7.0_NWA: 100%|██████████| 16/16 [00:01<00:00, 11.75it/s]


  ✓ [WZjj_QCD_v3.7.0_NWA] done -> Total=10000 events, passed=166

[1/2] WZjj_EW_v3.7.0_NWA: Processing 1 file(s) with 48 worker(s), 16 chunk(s) per file...


...

  ✓ [WZjj_QCD_v3.7.0_NWA] done -> Total=10000 events, passed=150

[1/2] WZjj_EW_v3.7.0_NWA: Processing 1 file(s) with 48 worker(s), 16 chunk(s) per file...



    WZjj_EW_v3.7.0_NWA: 100%|██████████| 16/16 [00:01<00:00, 13.31it/s]


  ✓ [WZjj_EW_v3.7.0_NWA] done -> Total=10000 events, passed=575

[2/2] WZjj_QCD_v3.7.0_NWA: Processing 1 file(s) with 48 worker(s), 16 chunk(s) per file...


    WZjj_QCD_v3.7.0_NWA: 100%|██████████| 16/16 [00:01<00:00, 11.95it/s]


  ✓ [WZjj_QCD_v3.7.0_NWA] done -> Total=10000 events, passed=224
MadGraph v3.7.0 NWA Expected Event Counts (L = 139 fb^-1)
Sample           Region   Initial XS (fb)    Total Events   Passed Events    Pass Rate   Final XS (fb) Expected Events (139/fb)
---------------------------------------------------------------------------------------------------------------------------------------
WZjj_EW_v3.7.0_NWA SR          2.880243e+00           10000             573        5.73%    1.650379e-01                    22.94
WZjj_QCD_v3.7.0_NWA SR          2.177010e+01           10000             166        1.66%    3.613836e-01                    50.23
---------------------------------------------------------------------------------------------------------------------------------------
SUM                                                                                                                       73.17


In [6]:
# === MadGraph 3.7.0 Full ===
print("\n" + "="*90)
print("COMPUTING: MadGraph v3.7.0 Full")
print("="*90)

if len(full_v370_avail) > 0:
    results_full_v370 = compute_expected_events_mg_by_region(
        sample_inputs=full_v370_avail,
        luminosity_fb_inv=luminosity_fb_inv,
        num_workers=num_workers,
        chunks_per_file=chunks_per_file,
    )
    print_expected_event_counts_table(
        results_full_v370,
        luminosity_fb_inv,
        title_prefix="MadGraph v3.7.0 Full Expected Event Counts",
        extended_mode=True,
        include_na=True,
    )
else:
    results_full_v370 = {}
    print("\n⚠ No available samples for v3.7.0 Full")


COMPUTING: MadGraph v3.7.0 Full
[Yields] Processing all samples with file + chunk parallelism...


[1/2] WZjj_EW_v3.7.0_full: Processing 1 file(s) with 48 worker(s), 16 chunk(s) per file...


    WZjj_EW_v3.7.0_full: 100%|██████████| 16/16 [00:01<00:00, 13.15it/s]


  ✓ [WZjj_EW_v3.7.0_full] done -> Total=10000 events, passed=567

[2/2] WZjj_QCD_v3.7.0_full: Processing 1 file(s) with 48 worker(s), 16 chunk(s) per file...


    WZjj_QCD_v3.7.0_full: 100%|██████████| 16/16 [00:01<00:00, 11.90it/s]


  ✓ [WZjj_QCD_v3.7.0_full] done -> Total=10000 events, passed=150

[1/2] WZjj_EW_v3.7.0_full: Processing 1 file(s) with 48 worker(s), 16 chunk(s) per file...


...

  ✓ [WZjj_QCD_v3.7.0_full] done -> Total=10000 events, passed=152

[1/2] WZjj_EW_v3.7.0_full: Processing 1 file(s) with 48 worker(s), 16 chunk(s) per file...


    WZjj_EW_v3.7.0_full: 100%|██████████| 16/16 [00:01<00:00, 13.31it/s]


  ✓ [WZjj_EW_v3.7.0_full] done -> Total=10000 events, passed=595

[2/2] WZjj_QCD_v3.7.0_full: Processing 1 file(s) with 48 worker(s), 16 chunk(s) per file...


    WZjj_QCD_v3.7.0_full: 100%|██████████| 16/16 [00:01<00:00, 11.89it/s]


  ✓ [WZjj_QCD_v3.7.0_full] done -> Total=10000 events, passed=250
MadGraph v3.7.0 Full Expected Event Counts (L = 139 fb^-1)
Sample           Region   Initial XS (fb)    Total Events   Passed Events    Pass Rate   Final XS (fb) Expected Events (139/fb)
---------------------------------------------------------------------------------------------------------------------------------------
WZjj_EW_v3.7.0_full SR          3.145628e+00           10000             567        5.67%    1.783571e-01                    24.79
WZjj_QCD_v3.7.0_full SR          2.357256e+01           10000             150        1.50%    3.535885e-01                    49.15
---------------------------------------------------------------------------------------------------------------------------------------
SUM                                                                                                                       73.94


## Comparison & Delta Analysis

In [9]:
# === Comparison Helper Function ===
def _extract_process_name(sample_key):
    """Normalize sample labels to process names for fair row-by-row comparison."""
    if "_v" in sample_key:
        return sample_key.split("_v", 1)[0]
    return sample_key


def _to_process_map(results_dict):
    """Map results keyed by sample label to results keyed by process name."""
    mapped = {}
    for sample_key, payload in results_dict.items():
        proc = _extract_process_name(sample_key)
        mapped[proc] = payload
    return mapped


def print_delta_comparison(results_baseline, results_new, baseline_label, new_label, region_key="n_expected_sr"):
    """Print comparison table with delta (absolute and percentage)."""
    baseline_map = _to_process_map(results_baseline)
    new_map = _to_process_map(results_new)

    print(f"\n{baseline_label} → {new_label} | Region: {region_key}")
    print("="*100)
    print(f"{'Process':<12} {'Baseline':>18} {'New':>18} {'Delta':>18} {'Delta %':>15}")
    print("-"*100)

    all_processes = set(baseline_map.keys()) | set(new_map.keys())

    for proc in sorted(all_processes):
        baseline_val = baseline_map.get(proc, {}).get(region_key, np.nan)
        new_val = new_map.get(proc, {}).get(region_key, np.nan)

        b_str = f"{baseline_val:.2f}" if np.isfinite(baseline_val) else "N/A"
        n_str = f"{new_val:.2f}" if np.isfinite(new_val) else "N/A"

        if np.isfinite(baseline_val) and np.isfinite(new_val):
            delta = new_val - baseline_val
            delta_pct = (delta / baseline_val * 100) if baseline_val != 0 else np.nan
            d_str = f"{delta:.2f}"
            dp_str = f"{delta_pct:.2f}%" if np.isfinite(delta_pct) else "N/A"
        else:
            d_str = "N/A"
            dp_str = "N/A"

        print(f"{proc:<12} {b_str:>18} {n_str:>18} {d_str:>18} {dp_str:>15}")

    print("="*100)


# === Approximation Comparison: NWA vs Full (v3.7.0) ===
print("\n" + "#"*100)
print("# APPROXIMATION COMPARISON: v3.7.0 NWA (baseline) vs v3.7.0 Full")
print("#"*100)

if len(full_v370_avail) > 0:
    print_delta_comparison(
        results_nwa_v370,
        results_full_v370,
        "v3.7.0 NWA",
        "v3.7.0 Full",
        region_key="n_expected_sr"
    )
else:
    print("\n⚠ v3.7.0 Full samples not available - cannot perform NWA vs Full comparison")


####################################################################################################
# APPROXIMATION COMPARISON: v3.7.0 NWA (baseline) vs v3.7.0 Full
####################################################################################################

v3.7.0 NWA → v3.7.0 Full | Region: n_expected_sr
Process                Baseline                New              Delta         Delta %
----------------------------------------------------------------------------------------------------
WZjj_EW                   22.94              24.79               1.85           8.07%
WZjj_QCD                  50.23              49.15              -1.08          -2.16%
